In [ ]:
from src.mta_graph import SubwayGraph
from src.complexes import ComplexesData
import pandas as pd

complexes = ComplexesData()
mta = SubwayGraph()

In [2]:
mta.build_graph()

In [ ]:
def get_onboards_data(complex_id: str, year: int = 2025, month: int = 9, day_of_week: str = "Monday", hour_of_day: str = "10"):
    from src.socrata_od_client import get_ridership_data

    # Get weekday ridership for May 2024 at 9am from station 623
    df = get_ridership_data(
        year=year,
        month=month,
        day_of_week=day_of_week,
        hour_of_day=hour_of_day,
        origin_station_complex_id=complex_id
    )

    df_ = df.sort_values('estimated_average_ridership', ascending=False)
    return df_

df_onboards_120 = get_onboards_data("120")
df_onboards_120

,year,month,day_of_week,hour_of_day,origin_station_complex_id,origin_station_complex_name,destination_station_complex_id,destination_station_complex_name,estimated_average_ridership
85,2025,9,Monday,10,120,Bedford Av (L),602,"14 St-Union Sq (L,N,Q,R,W,4,5,6)",130.6930
33,2025,9,Monday,10,120,Bedford Av (L),618,"14 St (A,C,E)/8 Av (L)",70.8448
167,2025,9,Monday,10,120,Bedford Av (L),601,"14 St (F,M,1,2,3)/6 Av (L)",43.5630
156,2025,9,Monday,10,120,Bedford Av (L),119,1 Av (L),42.7778
112,2025,9,Monday,10,120,Bedford Av (L),610,"Grand Central-42 St (S,4,5,6,7)",41.5626
...,...,...,...,...,...,...,...,...,...
198,2025,9,Monday,10,120,Bedford Av (L),421,"Gun Hill Rd (2,5)",0.2224
86,2025,9,Monday,10,120,Bedford Av (L),5,"36 Av (N,W)",0.2206
196,2025,9,Monday,10,120,Bedford Av (L),30,Prospect Av (R),0.2206
206,2025,9,Monday,10,120,Bedford Av (L),48,Avenue H (Q),0.2206


In [4]:
# 0 = north/east-bound, 1 = south/west-bound
def get_ordered_stops(line : str, direction: int, return_type : int = 0):
    complex_ids = [int(complexes.get_complex_id_by_gtfs_stop_id(stop[:-1])) for stop in mta.ordered_stops(line, direction)]
    complex_names = [complexes.get_station_name_by_gtfs_id(stop[:-1]) for stop in mta.ordered_stops(line, direction)]
    if return_type == 0:
        return dict(zip(complex_names, complex_ids))
    elif return_type == 1:
        return complex_ids
    else:
        return complex_names

get_ordered_stops("L", 0, 0)

{'Canarsie-Rockaway Pkwy': 138,
 'East 105 St': 137,
 'New Lots Av': 136,
 'Livonia Av': 135,
 'Sutter Av': 134,
 'Atlantic Av': 133,
 'Broadway Junction': 621,
 'Bushwick Av-Aberdeen St': 131,
 'Wilson Av': 130,
 'Halsey St': 129,
 'Myrtle-Wyckoff Avs': 630,
 'DeKalb Av': 127,
 'Jefferson St': 126,
 'Morgan Av': 125,
 'Montrose Av': 124,
 'Grand St': 123,
 'Graham Av': 122,
 'Lorimer St': 629,
 'Bedford Av': 120,
 '1 Av': 119,
 '3 Av': 118,
 '14 St-Union Sq': 602,
 '6 Av': 601,
 '8 Av': 618}

In [5]:
# 0 = north/east-bound, 1 = south/west-bound
def get_onboardings(origin_complex_id: str, line: str, direction: int, df_ : any):
    onboardings = 0
    boardings_df = pd.DataFrame(columns=df_.columns)
    stops_on_line = get_ordered_stops(line, direction, 1)
    stops_after = stops_on_line[stops_on_line.index(int(origin_complex_id)) + 1:]

    for i in range(len(df_)):
        print()
        destination_complex_id = str(df_.iloc[i]["destination_station_complex_id"])
        connecting_lines = mta.connecting_lines(origin_complex_id, destination_complex_id)
        
        print(f"Destination complex id: {destination_complex_id}")
        print(f"Connecting lines: {connecting_lines}")

        if len(connecting_lines) > 0: # Direct Connection Exists
            if line in connecting_lines:
                print(f"Stops after stop {origin_complex_id}: {stops_after}")

                if int(destination_complex_id) in stops_after:
                    print(f"{destination_complex_id} is in {stops_after}")
                    onboardings += df_.iloc[i]["estimated_average_ridership"] / len(connecting_lines)
                    df_tmp = df_.iloc[[i]].copy()
                    df_tmp["estimated_average_ridership"] /= len(connecting_lines)
                    boardings_df =  pd.concat([boardings_df, df_tmp], ignore_index=True)
                else:
                    print(f"{destination_complex_id} is not in stops_after")

        else: # Transfer Required
            shortest_paths = mta.all_shortest_paths(origin_complex_id, destination_complex_id)
            print(f"From {origin_complex_id} to {destination_complex_id}, shortest paths: {shortest_paths}")

            total_paths = 0
            num_paths = 0
            distances = {}
            for path in shortest_paths:
                connections = mta.connecting_lines(path[0], path[1])

                # total_distance = 0
                # all_connections = []
                # for j in range(len(path) - 1):
                #     connecting_line = mta.connecting_lines(path[j], path[j+1])[0]
                #     stops = get_ordered_stops(connecting_line, direction, 1)
                #     path_distance = abs(stops.index(int(path[j])) - stops.index(int(path[j+1])))
                #     all_connections += connecting_line
                #     total_distance += path_distance
                # distances[str(path)] = total_distance
                # print("Path: ", path, ", Total distance: ", total_distance, ", Connections: ", all_connections)
                # distance = abs(stops_on_line.index(int(path[0])) - stops_on_line.index(int(path[1])))
                # print(f"From {path[0]} to {path[1]}, connections: {connections}, distance: {distance}")
                total_paths += len(connections)
                if line in connections:
                    if int(path[1]) in stops_after:
                        num_paths += 1
            print(f"Total paths: {total_paths}, num paths: {num_paths}")
            print(i)
            if total_paths > 0 and num_paths > 0:
                onboardings += df_.iloc[i]["estimated_average_ridership"] * (num_paths / total_paths)
                df_tmp = df_.iloc[[i]].copy()
                df_tmp["estimated_average_ridership"] *= (num_paths / total_paths)
                print("df_tmp: ", df_tmp["destination_station_complex_id"])
                boardings_df =  pd.concat([boardings_df, df_tmp], ignore_index=True)

    return onboardings, boardings_df

In [6]:
df_onboards_120["estimated_average_ridership"].sum() # number of people who boarded the L train at Bedford Av at 9 am on Tuesday in Febuary 2025.

np.float64(1059.9913999999999)

In [7]:
# 0 = north/east-bound, 1 = south/west-bound
obs, df = get_onboardings ("120", "L", 0, df_onboards_120)

print("Onboardings: ", obs)
df


Destination complex id: 602
Connecting lines: ['L']
Stops after stop 120: [119, 118, 602, 601, 618]
602 is in [119, 118, 602, 601, 618]

Destination complex id: 618
Connecting lines: ['L']
Stops after stop 120: [119, 118, 602, 601, 618]
618 is in [119, 118, 602, 601, 618]

Destination complex id: 601
Connecting lines: ['L']
Stops after stop 120: [119, 118, 602, 601, 618]
601 is in [119, 118, 602, 601, 618]

Destination complex id: 119
Connecting lines: ['L']
Stops after stop 120: [119, 118, 602, 601, 618]
119 is in [119, 118, 602, 601, 618]

Destination complex id: 610
Connecting lines: []
From 120 to 610, shortest paths: [['120', '602', '610']]
Total paths: 1, num paths: 1
4
df_tmp:  112    610
Name: destination_station_complex_id, dtype: int64

Destination complex id: 164
Connecting lines: []
From 120 to 164, shortest paths: [['120', '618', '164'], ['120', '621', '164']]
Total paths: 2, num paths: 1
5
df_tmp:  219    164
Name: destination_station_complex_id, dtype: int64

Destinatio

/var/folders/vv/8v01rf7n71748vxd0m5kt_dr0000gn/T/ipykernel_97437/4148522952.py:25: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  boardings_df =  pd.concat([boardings_df, df_tmp], ignore_index=True)


df_tmp:  173    445
Name: destination_station_complex_id, dtype: int64

Destination complex id: 303
Connecting lines: []
From 120 to 303, shortest paths: [['120', '601', '303']]
Total paths: 1, num paths: 1
142
df_tmp:  32    303
Name: destination_station_complex_id, dtype: int64

Destination complex id: 278
Connecting lines: []
From 120 to 278, shortest paths: [['120', '618', '278'], ['120', '621', '278']]
Total paths: 2, num paths: 1
143
df_tmp:  102    278
Name: destination_station_complex_id, dtype: int64

Destination complex id: 109
Connecting lines: []
From 120 to 109, shortest paths: [['120', '630', '109']]
Total paths: 1, num paths: 0
144

Destination complex id: 392
Connecting lines: []
From 120 to 392, shortest paths: [['120', '602', '392']]
Total paths: 1, num paths: 1
145
df_tmp:  18    392
Name: destination_station_complex_id, dtype: int64

Destination complex id: 175
Connecting lines: []
From 120 to 175, shortest paths: [['120', '618', '175'], ['120', '629', '175'], ['120

,year,month,day_of_week,hour_of_day,origin_station_complex_id,origin_station_complex_name,destination_station_complex_id,destination_station_complex_name,estimated_average_ridership
0,2025,9,Monday,10,120,Bedford Av (L),602,"14 St-Union Sq (L,N,Q,R,W,4,5,6)",130.6930
1,2025,9,Monday,10,120,Bedford Av (L),618,"14 St (A,C,E)/8 Av (L)",70.8448
2,2025,9,Monday,10,120,Bedford Av (L),601,"14 St (F,M,1,2,3)/6 Av (L)",43.5630
3,2025,9,Monday,10,120,Bedford Av (L),119,1 Av (L),42.7778
4,2025,9,Monday,10,120,Bedford Av (L),610,"Grand Central-42 St (S,4,5,6,7)",41.5626
...,...,...,...,...,...,...,...,...,...
209,2025,9,Monday,10,120,Bedford Av (L),156,"103 St (C,B)",0.1112
210,2025,9,Monday,10,120,Bedford Av (L),421,"Gun Hill Rd (2,5)",0.2224
211,2025,9,Monday,10,120,Bedford Av (L),5,"36 Av (N,W)",0.2206
212,2025,9,Monday,10,120,Bedford Av (L),30,Prospect Av (R),0.2206


In [8]:
df["estimated_average_ridership"].sum() # number of people who boarded the L train at Bedford Av at 9 am on Tuesday in Febuary 2025.

np.float64(867.38757163385)

In [9]:
def get_offboards_data(complex_id, year: int = 2025, month: int = 9, day_of_week: str = "Monday", hour_of_day: str = "10"):
    from socrata_od_client import get_ridership_data

    # Get weekday ridership for May 2024 at 9am from station 623
    df = get_ridership_data(
        year=year,
        month=month,
        day_of_week=day_of_week,
        hour_of_day=hour_of_day,
        destination_station_complex_id=complex_id
    )

    df_ = df.sort_values('estimated_average_ridership', ascending=False)
    return df_

df_offboards_120 = get_offboards_data("120")
df_offboards_120

,year,month,day_of_week,hour_of_day,origin_station_complex_id,origin_station_complex_name,destination_station_complex_id,destination_station_complex_name,estimated_average_ridership
290,2025,9,Monday,10,119,1 Av (L),120,Bedford Av (L),61.3278
220,2025,9,Monday,10,602,"14 St-Union Sq (L,N,Q,R,W,4,5,6)",120,Bedford Av (L),42.7488
75,2025,9,Monday,10,630,"Myrtle-Wyckoff Avs (L,M)",120,Bedford Av (L),40.4088
70,2025,9,Monday,10,127,DeKalb Av (L),120,Bedford Av (L),39.3482
207,2025,9,Monday,10,611,"Times Sq-42 St (N,Q,R,W,S,1,2,3,7)/42 St (A,C,E)",120,Bedford Av (L),32.3374
...,...,...,...,...,...,...,...,...,...
249,2025,9,Monday,10,241,"15 St-Prospect Park (F,G)",120,Bedford Av (L),0.2228
325,2025,9,Monday,10,334,"Clark St (2,3)",120,Bedford Av (L),0.2208
226,2025,9,Monday,10,145,190 St (A),120,Bedford Av (L),0.2202
237,2025,9,Monday,10,303,157 St (1),120,Bedford Av (L),0.2196


In [10]:
# 0 = north/east-bound, 1 = south/west-bound
def get_offboardings(complex_id: str, line: str, direction: int, df_ : any):
    onboardings = 0
    boardings_df = pd.DataFrame(columns=df_.columns)
    stops_on_line = get_ordered_stops(line, direction, 1)

    for i in range(len(df_)):
        origin_complex_id = str(df_.iloc[i]["origin_station_complex_id"])
        connecting_lines = mta.connecting_lines(complex_id, origin_complex_id)
        
        print(f"Origin complex id: {origin_complex_id}, with name: ", mta.complex_id_to_name(origin_complex_id))
        print(f"Connecting lines: {connecting_lines}")

        if len(connecting_lines) > 0:
            if line in connecting_lines:
                stops_before = get_ordered_stops(line, 1 - direction, 1)
                stops_before = stops_before[stops_before.index(int(complex_id)) + 1:]

                print(f"Stops before stop {complex_id}: {stops_before}")

                if int(origin_complex_id) in stops_before:
                    print(f"{origin_complex_id} is in {stops_before}")
                    onboardings += df_.iloc[i]["estimated_average_ridership"] / len(connecting_lines)
                    df_tmp = df_.iloc[[i]].copy()
                    df_tmp["estimated_average_ridership"] /= len(connecting_lines)
                    boardings_df =  pd.concat([boardings_df, df_tmp], ignore_index=True)
                else:
                    print(f"{origin_complex_id} is not in stops_before")

        else:
            shortest_paths = mta.all_shortest_paths(complex_id, origin_complex_id)
            print(f"From {origin_complex_id} to {complex_id}, shortest paths: {shortest_paths}")

            total_paths = 0
            num_paths = 0
            distances = {}
            for path in shortest_paths:
                connections = mta.connecting_lines(path[0], path[1])

                # total_distance = 0
                # all_connections = []
                # for j in range(len(path) - 1):
                #     connecting_line = mta.connecting_lines(path[j], path[j+1])[0]
                #     stops = get_ordered_stops(connecting_line, direction, 1)
                #     path_distance = abs(stops.index(int(path[j])) - stops.index(int(path[j+1])))
                #     all_connections += connecting_line
                #     total_distance += path_distance
                # distances[str(path)] = total_distance
                # print("Path: ", path, ", Total distance: ", total_distance, ", Connections: ", all_connections)

                # distance = abs(stops_on_line.index(int(path[0])) - stops_on_line.index(int(path[1])))
                # print(f"From {path[0]} to {path[1]}, connections: {connections}, distance: {distance}")
                total_paths += len(connections)
                if line in connections:
                    if int(path[1]) in stops_before:
                        num_paths += 1
            print(f"Total paths: {total_paths}, num paths: {num_paths}")
            print(i)
            if total_paths > 0 and num_paths > 0:
                onboardings += df_.iloc[i]["estimated_average_ridership"] * (num_paths / total_paths)
                df_tmp = df_.iloc[[i]].copy()
                df_tmp["estimated_average_ridership"] *= (num_paths / total_paths)
                print("df_tmp: ", df_tmp["destination_station_complex_id"])
                boardings_df =  pd.concat([boardings_df, df_tmp], ignore_index=True)

    return onboardings, boardings_df


In [11]:
stops_on_line = get_ordered_stops("L", 1, 1)
print(stops_on_line)
shortest_paths = mta.all_shortest_paths("120", "611")
shortest_paths

[618, 601, 602, 118, 119, 120, 629, 122, 123, 124, 125, 126, 127, 630, 129, 130, 131, 621, 133, 134, 135, 136, 137, 138]


[['120', '602', '611'],
 ['120', '601', '611'],
 ['120', '618', '611'],
 ['120', '621', '611']]

In [12]:
for path in shortest_paths:
    connections = mta.connecting_lines(path[0], path[1])
    print(connections)
    distance = abs(stops_on_line.index(int(path[0])) - stops_on_line.index(int(path[1])))
    print(distance)

['L']
3
['L']
4
['L']
5
['L']
12


In [13]:
df_offboards_120[0:5]

,year,month,day_of_week,hour_of_day,origin_station_complex_id,origin_station_complex_name,destination_station_complex_id,destination_station_complex_name,estimated_average_ridership
290,2025,9,Monday,10,119,1 Av (L),120,Bedford Av (L),61.3278
220,2025,9,Monday,10,602,"14 St-Union Sq (L,N,Q,R,W,4,5,6)",120,Bedford Av (L),42.7488
75,2025,9,Monday,10,630,"Myrtle-Wyckoff Avs (L,M)",120,Bedford Av (L),40.4088
70,2025,9,Monday,10,127,DeKalb Av (L),120,Bedford Av (L),39.3482
207,2025,9,Monday,10,611,"Times Sq-42 St (N,Q,R,W,S,1,2,3,7)/42 St (A,C,E)",120,Bedford Av (L),32.3374


In [14]:
# 0 = north/east-bound (towards 8 Av), 1 = south/west-bound (towards Rockaway)
obs, df = get_offboardings ("120", "L", 1, df_offboards_120)

print("Offboardings: ", obs)
df

Origin complex id: 119, with name:  1 Av
Connecting lines: ['L']
Stops before stop 120: [119, 118, 602, 601, 618]
119 is in [119, 118, 602, 601, 618]
Origin complex id: 602, with name:  14 St-Union Sq
Connecting lines: ['L']
Stops before stop 120: [119, 118, 602, 601, 618]
602 is in [119, 118, 602, 601, 618]
Origin complex id: 630, with name:  Myrtle-Wyckoff Avs
Connecting lines: ['L']
Stops before stop 120: [119, 118, 602, 601, 618]
630 is not in stops_before
Origin complex id: 127, with name:  DeKalb Av
Connecting lines: ['L']
Stops before stop 120: [119, 118, 602, 601, 618]
127 is not in stops_before
Origin complex id: 611, with name:  Times Sq-42 St
Connecting lines: []
From 611 to 120, shortest paths: [['120', '602', '611'], ['120', '601', '611'], ['120', '618', '611'], ['120', '621', '611']]
Total paths: 4, num paths: 3
4
df_tmp:  207    120
Name: destination_station_complex_id, dtype: int64
Origin complex id: 601, with name:  14 St
Connecting lines: ['L']
Stops before stop 120: 

/var/folders/vv/8v01rf7n71748vxd0m5kt_dr0000gn/T/ipykernel_97437/1818246956.py:26: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  boardings_df =  pd.concat([boardings_df, df_tmp], ignore_index=True)


Stops before stop 120: [119, 118, 602, 601, 618]
138 is not in stops_before
Origin complex id: 397, with name:  86 St
Connecting lines: []
From 397 to 120, shortest paths: [['120', '602', '397']]
Total paths: 1, num paths: 1
20
df_tmp:  95    120
Name: destination_station_complex_id, dtype: int64
Origin complex id: 628, with name:  Fulton St
Connecting lines: []
From 628 to 120, shortest paths: [['120', '602', '628'], ['120', '601', '628'], ['120', '618', '628'], ['120', '621', '628']]
Total paths: 4, num paths: 3
21
df_tmp:  32    120
Name: destination_station_complex_id, dtype: int64
Origin complex id: 131, with name:  Bushwick Av-Aberdeen St
Connecting lines: ['L']
Stops before stop 120: [119, 118, 602, 601, 618]
131 is not in stops_before
Origin complex id: 612, with name:  51 St
Connecting lines: []
From 612 to 120, shortest paths: [['120', '602', '612'], ['120', '618', '612']]
Total paths: 2, num paths: 2
23
df_tmp:  256    120
Name: destination_station_complex_id, dtype: int64
O

,year,month,day_of_week,hour_of_day,origin_station_complex_id,origin_station_complex_name,destination_station_complex_id,destination_station_complex_name,estimated_average_ridership
0,2025,9,Monday,10,119,1 Av (L),120,Bedford Av (L),61.32780
1,2025,9,Monday,10,602,"14 St-Union Sq (L,N,Q,R,W,4,5,6)",120,Bedford Av (L),42.74880
2,2025,9,Monday,10,611,"Times Sq-42 St (N,Q,R,W,S,1,2,3,7)/42 St (A,C,E)",120,Bedford Av (L),24.25305
3,2025,9,Monday,10,601,"14 St (F,M,1,2,3)/6 Av (L)",120,Bedford Av (L),32.01140
4,2025,9,Monday,10,164,"34 St-Penn Station (A,C,E)",120,Bedford Av (L),14.38670
...,...,...,...,...,...,...,...,...,...
269,2025,9,Monday,10,241,"15 St-Prospect Park (F,G)",120,Bedford Av (L),0.11140
270,2025,9,Monday,10,334,"Clark St (2,3)",120,Bedford Av (L),0.22080
271,2025,9,Monday,10,145,190 St (A),120,Bedford Av (L),0.11010
272,2025,9,Monday,10,303,157 St (1),120,Bedford Av (L),0.21960


In [15]:
# 0 = north/east-bound, 1 = south/west-bound
print(get_ordered_stops("L", 1))

{'8 Av': 618, '6 Av': 601, '14 St-Union Sq': 602, '3 Av': 118, '1 Av': 119, 'Bedford Av': 120, 'Lorimer St': 629, 'Graham Av': 122, 'Grand St': 123, 'Montrose Av': 124, 'Morgan Av': 125, 'Jefferson St': 126, 'DeKalb Av': 127, 'Myrtle-Wyckoff Avs': 630, 'Halsey St': 129, 'Wilson Av': 130, 'Bushwick Av-Aberdeen St': 131, 'Broadway Junction': 621, 'Atlantic Av': 133, 'Sutter Av': 134, 'Livonia Av': 135, 'New Lots Av': 136, 'East 105 St': 137, 'Canarsie-Rockaway Pkwy': 138}


In [16]:
from socrata_od_client import get_ridership_data

complex_id = "118"
line = "L"
direction = 1 # 0 = north/east-bound (towards 8 Av), 1 = south/west-bound (towards Rockaway)

year = 2025
month = 9
day_of_week = "Monday"
hour_of_day = "9"

# Get weekday ridership for May 2024 at 9am from station 623
df_onboards = get_onboards_data(complex_id, year, month, day_of_week, hour_of_day)
df_offboards = get_offboards_data(complex_id, year, month, day_of_week, hour_of_day)


obs_onboard, df_onboard = get_onboardings (complex_id, line, direction, df_onboards)
obs_offboard, df_offboard = get_offboardings (complex_id, line, direction, df_offboards)



Destination complex id: 120
Connecting lines: ['L']
Stops after stop 118: [119, 120, 629, 122, 123, 124, 125, 126, 127, 630, 129, 130, 131, 621, 133, 134, 135, 136, 137, 138]
120 is in [119, 120, 629, 122, 123, 124, 125, 126, 127, 630, 129, 130, 131, 621, 133, 134, 135, 136, 137, 138]

Destination complex id: 618
Connecting lines: ['L']
Stops after stop 118: [119, 120, 629, 122, 123, 124, 125, 126, 127, 630, 129, 130, 131, 621, 133, 134, 135, 136, 137, 138]
618 is not in stops_after

Destination complex id: 122
Connecting lines: ['L']
Stops after stop 118: [119, 120, 629, 122, 123, 124, 125, 126, 127, 630, 129, 130, 131, 621, 133, 134, 135, 136, 137, 138]
122 is in [119, 120, 629, 122, 123, 124, 125, 126, 127, 630, 129, 130, 131, 621, 133, 134, 135, 136, 137, 138]

Destination complex id: 164
Connecting lines: []
From 118 to 164, shortest paths: [['118', '618', '164'], ['118', '621', '164']]
Total paths: 2, num paths: 1
3
df_tmp:  35    164
Name: destination_station_complex_id, dtype:

/var/folders/vv/8v01rf7n71748vxd0m5kt_dr0000gn/T/ipykernel_97437/4148522952.py:25: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  boardings_df =  pd.concat([boardings_df, df_tmp], ignore_index=True)


Stops before stop 118: [602, 601, 618]
127 is not in stops_before
Origin complex id: 122, with name:  Graham Av
Connecting lines: ['L']
Stops before stop 118: [602, 601, 618]
122 is not in stops_before
Origin complex id: 124, with name:  Montrose Av
Connecting lines: ['L']
Stops before stop 118: [602, 601, 618]
124 is not in stops_before
Origin complex id: 126, with name:  Jefferson St
Connecting lines: ['L']
Stops before stop 118: [602, 601, 618]
126 is not in stops_before
Origin complex id: 125, with name:  Morgan Av
Connecting lines: ['L']
Stops before stop 118: [602, 601, 618]
125 is not in stops_before
Origin complex id: 129, with name:  Halsey St
Connecting lines: ['L']
Stops before stop 118: [602, 601, 618]
129 is not in stops_before
Origin complex id: 283, with name:  Greenpoint Av
Connecting lines: []
From 283 to 118, shortest paths: [['118', '629', '283']]
Total paths: 1, num paths: 0
9
Origin complex id: 289, with name:  Bedford-Nostrand Avs
Connecting lines: []
From 289 to 

/var/folders/vv/8v01rf7n71748vxd0m5kt_dr0000gn/T/ipykernel_97437/1818246956.py:64: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  boardings_df =  pd.concat([boardings_df, df_tmp], ignore_index=True)


Stops before stop 118: [602, 601, 618]
119 is not in stops_before
Origin complex id: 71, with name:  8 Av
Connecting lines: []
From 71 to 118, shortest paths: [['118', '602', '71']]
Total paths: 1, num paths: 1
30
df_tmp:  114    118
Name: destination_station_complex_id, dtype: int64
Origin complex id: 607, with name:  34 St-Herald Sq
Connecting lines: []
From 607 to 118, shortest paths: [['118', '602', '607'], ['118', '601', '607']]
Total paths: 2, num paths: 2
31
df_tmp:  115    118
Name: destination_station_complex_id, dtype: int64
Origin complex id: 195, with name:  Ozone Park-Lefferts Blvd
Connecting lines: []
From 195 to 118, shortest paths: [['118', '618', '195'], ['118', '621', '195']]
Total paths: 2, num paths: 1
32
df_tmp:  10    118
Name: destination_station_complex_id, dtype: int64
Origin complex id: 83, with name:  Woodhaven Blvd
Connecting lines: []
From 83 to 118, shortest paths: [['118', '621', '83']]
Total paths: 1, num paths: 0
33
Origin complex id: 165, with name:  2

In [18]:
print("Total Onboardings: ", obs_onboard)

df_onboard.head(15)

Total Onboardings:  65.7372208699902


,year,month,day_of_week,hour_of_day,origin_station_complex_id,origin_station_complex_name,destination_station_complex_id,destination_station_complex_name,estimated_average_ridership
0,2025,9,Monday,9,118,3 Av (L),120,Bedford Av (L),21.71060
1,2025,9,Monday,9,118,3 Av (L),122,Graham Av (L),5.35300
2,2025,9,Monday,9,118,3 Av (L),164,"34 St-Penn Station (A,C,E)",2.50580
3,2025,9,Monday,9,118,3 Av (L),629,Lorimer St (L)/Metropolitan Av (G),3.75920
4,2025,9,Monday,9,118,3 Av (L),611,"Times Sq-42 St (N,Q,R,W,S,1,2,3,7)/42 St (A,C,E)",0.78825
5,2025,9,Monday,9,118,3 Av (L),138,Canarsie-Rockaway Pkwy (L),2.62560
6,2025,9,Monday,9,118,3 Av (L),283,Greenpoint Av (G),2.56320
7,2025,9,Monday,9,118,3 Av (L),168,"Spring St (C,E)",1.14320
8,2025,9,Monday,9,118,3 Av (L),127,DeKalb Av (L),2.28160
9,2025,9,Monday,9,118,3 Av (L),129,Halsey St (L),1.44080


In [111]:
print("Total Offboardings: ", obs_offboard)

df_offboard.head(15)

Total Offboardings:  880.678124900392


,year,month,day_of_week,hour_of_day,origin_station_complex_id,origin_station_complex_name,destination_station_complex_id,destination_station_complex_name,estimated_average_ridership
0,2025,9,Tuesday,9,120,Bedford Av (L),618,"14 St (A,C,E)/8 Av (L)",185.263600
1,2025,9,Tuesday,9,610,"Grand Central-42 St (S,4,5,6,7)",618,"14 St (A,C,E)/8 Av (L)",17.579778
2,2025,9,Tuesday,9,629,Lorimer St (L)/Metropolitan Av (G),618,"14 St (A,C,E)/8 Av (L)",74.554000
3,2025,9,Tuesday,9,119,1 Av (L),618,"14 St (A,C,E)/8 Av (L)",65.635400
4,2025,9,Tuesday,9,630,"Myrtle-Wyckoff Avs (L,M)",618,"14 St (A,C,E)/8 Av (L)",49.203400
5,2025,9,Tuesday,9,122,Graham Av (L),618,"14 St (A,C,E)/8 Av (L)",49.163800
6,2025,9,Tuesday,9,127,DeKalb Av (L),618,"14 St (A,C,E)/8 Av (L)",41.208400
7,2025,9,Tuesday,9,129,Halsey St (L),618,"14 St (A,C,E)/8 Av (L)",30.318200
8,2025,9,Tuesday,9,125,Morgan Av (L),618,"14 St (A,C,E)/8 Av (L)",28.248600
9,2025,9,Tuesday,9,124,Montrose Av (L),618,"14 St (A,C,E)/8 Av (L)",27.230200


In [96]:
stops_on_line = get_ordered_stops(line, direction, 1)
stops_on_line

[618,
 601,
 602,
 118,
 119,
 120,
 629,
 122,
 123,
 124,
 125,
 126,
 127,
 630,
 129,
 130,
 131,
 621,
 133,
 134,
 135,
 136,
 137,
 138]

In [93]:
df_onboards.head(15)

,year,month,day_of_week,hour_of_day,origin_station_complex_id,origin_station_complex_name,destination_station_complex_id,destination_station_complex_name,estimated_average_ridership
37,2025,2,Tuesday,9,618,"14 St (A,C,E)/8 Av (L)",164,"34 St-Penn Station (A,C,E)",89.6797
83,2025,2,Tuesday,9,618,"14 St (A,C,E)/8 Av (L)",276,"5 Av/53 St (E,M)",75.4958
85,2025,2,Tuesday,9,618,"14 St (A,C,E)/8 Av (L)",611,"Times Sq-42 St (N,Q,R,W,S,1,2,3,7)/42 St (A,C,E)",57.8625
38,2025,2,Tuesday,9,618,"14 St (A,C,E)/8 Av (L)",624,"Chambers St (A,C)/WTC (E)/Park Pl (2,3)/Cortla...",49.6112
198,2025,2,Tuesday,9,618,"14 St (A,C,E)/8 Av (L)",610,"Grand Central-42 St (S,4,5,6,7)",45.9287
183,2025,2,Tuesday,9,618,"14 St (A,C,E)/8 Av (L)",612,"Lexington Av-53 St (E,M)/51 St (6)",44.2830
166,2025,2,Tuesday,9,618,"14 St (A,C,E)/8 Av (L)",168,"Spring St (C,E)",36.6125
72,2025,2,Tuesday,9,618,"14 St (A,C,E)/8 Av (L)",614,"59 St-Columbus Circle (A,B,C,D,1)",32.9348
241,2025,2,Tuesday,9,618,"14 St (A,C,E)/8 Av (L)",169,"Canal St (A,C,E)",31.7987
127,2025,2,Tuesday,9,618,"14 St (A,C,E)/8 Av (L)",628,"Fulton St (A,C,J,Z,2,3,4,5)",27.2395


In [89]:
df_onboards = get_onboards_data(complex_id, year, month, day_of_week, hour_of_day)
obs, df = get_onboardings (complex_id, line, line, df_onboards)


L
L


IndexError: single positional indexer is out-of-bounds